# 08 --- Integration: Full End-to-End Research Query

This notebook brings together all CCA patterns:
- Hub-and-spoke coordinator
- Context isolation via explicit passing
- 4-5 scoped tools per agent
- Structured error handling
- Parallel + sequential task waves
- Deterministic conflict resolution

We'll walk through a complete research query using the `economic_impact` scenario.
Four scripted subagents each call real tools through the real loop; one fetch
times out and the fact checker flags a contradiction, so every step of the
6-step flow has something to do.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
import json

from research_agents.models.research import SubTask
from research_agents.agent.coordinator import (
    build_research_report, collect_gaps, collect_tool_errors, run_coordinator, sort_tasks_into_waves,
)
from research_agents.agent.context_builder import build_subagent_context
from research_agents.anti_patterns.silent_failures import silent_dispatch
from research_agents.data.scenarios import SCENARIOS
from research_agents.data.sources import SOURCE_RELIABILITY_RATINGS
from research_agents.services.container import make_default_services
from research_agents.testing import scripted_client, text_turn, tool_turn
from research_agents.tools.handlers import dispatch

## The ResearchScenario Model

The project defines pre-built scenarios (in `data/scenarios.py`) that combine
all the patterns we've studied:

```python
@dataclass(frozen=True)
class ResearchScenario:
    name: str
    query: str
    expected_agents: list[str]  # Which agent types should be involved
    expected_conflicts: int     # How many source contradictions
    expected_gaps: int          # How many sources will fail
    description: str
```

Three scenarios, each targeting different patterns:

| Scenario | Tests | Conflicts | Gaps |
|----------|-------|-----------|------|
| `climate_renewable` | Conflict resolution | 1 (blog vs .gov) | 0 |
| `ai_healthcare` | Error handling | 0 | 1 (404) |
| `economic_impact` | Both | 1 (+13% vs -20%) | 1 (timeout) |

## Scenario: Remote Work Economic Impact

This scenario tests both conflict resolution AND error handling.

In [ ]:
scenario = SCENARIOS['economic_impact']
print(f'Query: {scenario.query}')
print(f'Expected agents: {scenario.expected_agents}')
print(f'Expected conflicts: {scenario.expected_conflicts}')
print(f'Expected gaps: {scenario.expected_gaps}')
print(f'Description: {scenario.description}')

## Step 1: Task Decomposition (PLAN + SORT)

The coordinator decomposes the query into SubTasks. In production the LLM does
this; the package leaves it to the caller. Here we define them explicitly to
show the structure:

In [ ]:
# Decompose into subtasks (normally the LLM does this)
tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='Search for studies on remote work and productivity',
        context='Focus on 2024 data. Look for both pro and con evidence.'),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='Query remote work statistics from the database',
        context='Use the remote_work_stats table'),
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='Analyze the Stanford remote work study',
        context='Document ID: doc-remote-work-stanford'),
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='Verify productivity claims and flag contradictions',
        context='Check claims about remote work productivity impact',
        depends_on=['web', 'data', 'docs']),
]

waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    print(f'Wave {i}: {[t.task_id for t in wave]} ({"parallel" if len(wave) > 1 else "sequential"})')

## Step 2: Context Isolation Check (DELEGATE)

Each subagent gets ONLY its explicit context. Wave-0 tasks have no
predecessors, so their context is just instruction + context. (The `facts`
task will also receive its three predecessors' results at run time; the
coordinator adds those, filtered by `depends_on`.)

In [ ]:
for task in tasks:
    ctx = build_subagent_context(task)
    print(f'{task.task_id} ({task.agent_type}):')
    print(f'  Context length: {len(ctx)} chars')
    print(f'  First 100 chars: {ctx[:100]}...')
    print()

## Step 3: Run Coordinator (scripted transcripts)

No API call is made. Each agent type gets a scripted transcript -- the tool
calls a model would make, then its summary -- and `run_coordinator()` does
everything else for real: context building, scoped tools, dispatch, tool
results, message history. The transcript is chosen per call from the *tools*
the coordinator passed, which is itself a check that each agent received its
own scoped set.

```python
def run_coordinator(client, services, tasks, model, max_iterations, dispatch_fn):
    waves = sort_tasks_into_waves(tasks)
    for wave in waves:
        for task in wave:
            config = SUBAGENT_CONFIGS[task.agent_type]
            context = build_subagent_context(task, results)
            result = run_agent_loop(client, services, context, ..., dispatch_fn=dispatch_fn)
            results[task.task_id] = result
    return results, waves
```

Note the web researcher's script: it fetches the timeout URL and then writes a
summary that mentions nothing about a failure. The report must not trust that.

In [ ]:
TIMEOUT_URL = 'https://timeout.example.com/remote-data'
MCKINSEY = 'https://mckinsey.com/future-of-work'
BLS = 'https://bls.gov/remote-work-stats'
BLOG = 'https://workfromhome-blog.example.com/productivity'

transcripts = {
    'web_researcher': [
        tool_turn('search_web', {'query': 'remote work economic impact on productivity'}),
        tool_turn('fetch_page', {'url': TIMEOUT_URL}),
        text_turn('Found 4 sources. McKinsey reports +13% productivity; a blog claims -20%.'),
    ],
    'data_extractor': [
        tool_turn('query_database', {'table': 'remote_work_stats'}),
        text_turn('Remote work stats: 9.4% fully remote, 18.2% hybrid in 2024.'),
    ],
    'document_analyzer': [
        tool_turn('parse_document', {'doc_id': 'doc-remote-work-stanford'}),
        text_turn('Stanford study: hybrid workers 13% more productive, 24% higher satisfaction.'),
    ],
    'fact_checker': [
        tool_turn('verify_claim', {'claim': 'Remote workers are 20% less productive'}),
        tool_turn('flag_conflict', {'claim': 'Remote workers are more productive',
                                    'sources_for': [MCKINSEY, BLS],
                                    'sources_against': [BLOG]}),
        text_turn('Verified: +13% claim supported. -20% claim contradicted by the knowledge base.'),
    ],
}

services = make_default_services()
client = scripted_client(transcripts)
results, waves = run_coordinator(client, services, tasks)

for task_id, result in results.items():
    print(f'{task_id}: {result.tool_calls} tool call(s), stop_reason={result.stop_reason}')
    print(f'   {result.content}')
print()
print(f'Tools per API call: {sorted({len(c["tools"]) for c in client.calls})}')

## Step 4: EVALUATE -- what the fact checker's tools returned

The fact checker is the EVALUATE step: it runs in wave 1 with the others'
results in its context, verifies a claim against the knowledge base, and flags
the contradiction. Its tool results are the coordinator's input for RESOLVE.
Note the `verify_claim` verdict: the blog's -20% claim comes back
`verified: false` because the knowledge base ranks the closest matching
record, not the most confident one.

In [ ]:
for entry in results['facts'].tool_results:
    data = json.loads(entry['result'])['data']
    print(f"{entry['tool_name']}: {json.dumps(data)[:150]}...")

def conflicts_from(results) -> list[dict]:
    """Conflicts the fact checker flagged, read from its tool results."""
    return [
        {k: json.loads(e['result'])['data'][k] for k in ('claim', 'sources_for', 'sources_against')}
        for e in results['facts'].tool_results
        if e['tool_name'] == 'flag_conflict'
    ]

conflicts = conflicts_from(results)
print()
print(f'Conflicts flagged: {len(conflicts)}')
print(f'Tool errors seen by the coordinator: {collect_tool_errors(results)}')
print(f'Gaps derived from them: {collect_gaps(results)}')

## Steps 5-6: Conflict Resolution + Report (RESOLVE + SYNTHESIZE)

The `build_research_report()` function compiles findings, resolves conflicts,
and produces a `ResearchReport`:

```python
class ResearchReport(BaseModel):
    query: str
    findings: list[SourceResult]       # One per subagent transcript
    conflicts: list[ConflictRecord]     # What contradicted, and which side won
    synthesis: str                      # Combined narrative
    confidence_score: float             # 0.0-1.0, adjusted by gaps/conflicts
    gaps: list[str]                     # What we couldn't reach (from tool errors)
```

`gaps` is not passed in below: the report derives it from the structured tool
errors. `SOURCE_RELIABILITY_RATINGS` is keyed by domain and the conflict's
sources are full URLs; the resolver matches them.

In [ ]:
report = build_research_report(
    query=scenario.query,
    results=results,
    reliability_lookup=SOURCE_RELIABILITY_RATINGS,
    conflicts=conflicts,
)

print(f'Query: {report.query}')
print(f'Findings: {len(report.findings)}')
print(f'Conflicts resolved: {len(report.conflicts)}')
for c in report.conflicts:
    print(f'  - {c.claim}: {c.resolution}, winning side = {c.winning_side} (confidence: {c.confidence:.2f})')
print(f'Gaps: {report.gaps}')
print(f'Confidence score: {report.confidence_score:.2f}')

## Comparison: the same run through the silent-failure router

Every transcript is replayed unchanged; only the router the coordinator
injects into the loop differs. The tool call counts and the conflict
resolution come out identical (the silent router only hides web fetch
failures), so the one variable is whether the timeout is *visible*.

In [ ]:
from helpers import compare_results

silent_results, _ = run_coordinator(scripted_client(transcripts), make_default_services(), tasks,
                                    dispatch_fn=silent_dispatch)
silent_report = build_research_report(scenario.query, silent_results, SOURCE_RELIABILITY_RATINGS,
                                      conflicts=conflicts_from(silent_results))

compare_results(
    {'tool_calls': sum(r.tool_calls for r in silent_results.values()),
     'timeout_visible_to_coordinator': len(collect_tool_errors(silent_results)) > 0,
     'gaps_reported': len(silent_report.gaps),
     'conflict_resolution': silent_report.conflicts[0].resolution,
     'confidence_score': silent_report.confidence_score},
    {'tool_calls': sum(r.tool_calls for r in results.values()),
     'timeout_visible_to_coordinator': len(collect_tool_errors(results)) > 0,
     'gaps_reported': len(report.gaps),
     'conflict_resolution': report.conflicts[0].resolution,
     'confidence_score': report.confidence_score},
)

## How the Pieces Connect

Here's the complete data flow we just executed:

```
ResearchQuery("remote work productivity")
  |                                              CCA Domain
  v                                              ---------
  1. PLAN: Decompose into 4 SubTasks             Agentic Architecture
  |    (web, data, docs, facts) -- by the caller
  v
  2. SORT: Topological sort into 2 waves          Agentic Architecture
  |    Wave 0: [web, data, docs] (parallel)
  |    Wave 1: [facts] (depends on web, data, docs)
  v
  3. DELEGATE: For each task:                     Context Management
  |    build_subagent_context() -> explicit string
  |    run_agent_loop() with scoped tools          Tool Design
  |    tool errors logged in AgentResult.tool_results
  v
  4. EVALUATE: fact_checker verifies + flags       Reliability
  |    flag_conflict results -> conflicts
  v
  5. RESOLVE: conflict_resolver.py                 Reliability
  |    reliability ranking -> majority -> human, with winning_side
  v
  6. SYNTHESIZE: ResearchReport                    All domains
       findings + conflicts + gaps (from tool errors) + confidence
```

## The Three Exam Walkthrough Questions

The published article walks through three representative CCA exam questions
drawn from the multi-agent research scenario. You have now seen every pattern
they test. Use these as cold-read self-checks.

Each question below lists the scenario, the correct answer (with a pointer to
the notebook that demonstrates it), and the distractors you'll see on the
actual exam along with *why* each distractor is wrong.

### Question 1 -- Context Isolation

**Scenario.** A user instructs the coordinator to "use APA citation format."
The web-research subagent returns sources formatted in MLA. Why?

**Correct answer.** The APA instruction lived in the coordinator's message
history but was never placed into the subagent's `task.context`. The
subagent structurally never saw it. The fix is to include formatting
requirements in every `SubTask.context` where they apply.

**Where to see it.** Notebook 02 (`02_context_isolation.ipynb`) -- three
subagents run through the real loop with a recording client: the leaky
`run_leaky_subagent`, the "forgot to forward" case, and the explicit
`build_subagent_context(task)` fix. The comparison table reads what each was
actually sent.

**Distractors and why they fail:**

- *"The subagent needs a better system prompt."* The system prompt is a
general persona, not a place for query-specific requirements. Putting APA in
every system prompt ever written is not context engineering.
- *"Use a larger model for the subagent."* A larger model that still never
sees the instruction will make the same error more fluently.
- *"Let subagents inherit coordinator context automatically."* This is the
`shared_context.py` anti-pattern -- it creates token waste, attention
dilution, and privacy-style leakage of other agents' results.

### Question 2 -- Tool Overload

**Scenario.** An agent configured with 18 tools repeatedly selects the wrong
tool for the task. What do you change?

**Correct answer.** Decompose the single agent into specialized subagents
with 4-5 focused tools each. This is an **architectural** fix -- it is not
a better-descriptions problem.

**Where to see it.** Notebook 03 (`03_tool_scoping.ipynb`) -- lists
`SUPER_AGENT_TOOLS` (20 on one agent) against the four subagents' sets (4
each), then replays one out-of-scope tool call through both routers: the
super agent executes it, scoped `dispatch` refuses it.
`tests/test_anti_patterns.py` asserts the super-agent count is 20;
`tests/test_tools.py` asserts the focused sets are exactly 4.

**Distractors and why they fail:**

- *"Improve the tool descriptions."* Canonical trap. Better descriptions on
18 overlapping tools still cause attention fragmentation; the agent spends
context budget evaluating descriptions rather than executing.
- *"Add a tool-selection preprocessing step."* You have added a second
agent to hide the first agent's problem. Now you have two agents to debug
and the underlying attention-fragmentation still exists during selection.
- *"Raise the temperature so the agent is less deterministic."* Actively
harmful -- tool selection should be *more* deterministic, not less.

### Question 3 -- Silent Failure

**Scenario.** A research report is missing a critical source after an
upstream API timeout. No error was flagged in the pipeline. What do you
change?

**Correct answer.** Require **structured error context** from subagents.
Every tool handler returns a `ToolErrorResponse` with `error_type`,
`retry_eligible`, `fallback_available`, and `source`. This gives the
coordinator a decision tree: retry transient failures, fallback for
permanent errors, flag gaps in the final report otherwise.

**Where to see it.** Notebook 04 (`04_error_handling.ipynb`) --
`handle_fetch_page_silent` vs the structured `dispatch` on the same timeout
URL, the 404 URL which produces a *different* decision tree, and the cascade
cell where the same transcript yields `gaps=[]` through the silent router and a
named gap through the structured one. The comparison at the end of this
notebook is the same experiment on the full four-agent run. Verified in
`tests/test_error_handling.py` and `tests/test_coordinator.py`.

**Distractors and why they fail:**

- *"Increase timeout duration."* Symptom fix. The next slow service still
fails silently; you have just moved the threshold.
- *"Add retry logic."* Partially correct -- retry is *one branch* of the
decision tree. It does not help when the failure is a 404 or a permanent
auth error. You cannot build the full decision tree on a response that
refuses to admit failure.
- *"Let the coordinator infer failures from response shape."* Requires the
coordinator to know every subagent's internal contract. Brittle. The
`ToolErrorResponse` schema makes the contract explicit and uniform.

## CCA Exam Tip

> The Multi-Agent Research System scenario draws from the three heaviest domains:
> - Agentic Architecture (27%)
> - Tool Design & MCP (18%)
> - Context Management & Reliability (15%)
>
> Together: **60% of the exam weight**. Master these patterns and you have a
framework for every scenario.
>
> Key models to know:
> - `SubTask` -- the unit of delegation with explicit context
> - `ToolErrorResponse` -- structured errors for informed decision-making
> - `ConflictRecord` -- deterministic resolution metadata, including which side won
> - `ResearchReport` -- transparent output with gaps and confidence